# Snapshot eval violins — seed comparison across sources

Compares **per-episode eval metrics** of several *sources* at each eval seed, drawn side by
side. Combines `sps_eval_violins_seeds.ipynb` (seed fan-out on the x-axis) with
`sps_eval_violins.ipynb` (coloured groups): every seed slot carries **one violin per source**,
so a seed shows each algorithm's violin next to the rule-based baseline's on that same
held-out draw.

A **source** is one snapshot under one eval kind (plus a rule-based strategy for `eval-rbc`).
`SOURCES` holds four here: SAC, PPO and DreamerV3 (`eval-ray`) against the
`price_median_scaled_0.25` baseline (`eval-rbc`), all from the `sta_lin_battery_only_price`
trial.

| `eval-kind` | Run directory pattern | Producer | Per-run CSV | Seed metadata |
|---|---|---|---|---|
| `eval-ray` | `runs/eval_<ts>[_seed{N}]/` | Ray RLlib eval (`run_eval_ray.py`) | `eval_results/*_eval/eval_*.csv` | sidecar JSON `"seed"` |
| `eval-sb` | `runs/eval_sb_<ts>[_seed{N}]/` | SB3 eval (`run_eval_sb.py`) | `eval_results/*_eval/eval_*.csv` | sidecar JSON `"seed"` |
| `eval-rbc` | `runs/eval_rbc_<ts>[_seed{N}]/` | Rule-based eval (`run_eval_rule_based.py`) | `eval_results/*_rule_based/<strategy>/episodes.csv` | `args.json` `"resolved_seed"` |

The actual seed a run used is read from that run's own metadata — never parsed from the
directory name — so the un-suffixed default run still gets its correct `trial.seed` label.
Produce the fan-outs with `./submit_snapshot_eval_seeds.sh <sweep.yaml>` (repo root), which
submits one `tools.snapshot.submit_snapshot --seed N` job per (snapshot, seed) pair.

**Main inputs:** `SOURCES` and `EXCLUDE_SEEDS` in the next cell — the latter removes seeds from
every figure and table; five are dropped here, leaving seeds 42, 45, 71, 91, 103 and 141.

In [101]:
# --- Main input ---------------------------------------------------------------------
# The two (or more) sources to compare side by side at each seed. Each source is one
# snapshot evaluated under one eval kind; at every seed the x-axis draws one violin per
# source. Keys per entry:
#   path          snapshot dir (absolute, or relative to the repo root)
#   eval-kind     "eval-ray" | "eval-sb" | "eval-rbc" — which runs/eval_* fan-out to use
#   label         short name shown in the legend and used as the per-seed violin group
#   color         violin fill colour for this source
#   rbc-strategy  (eval-rbc only) which rule-based strategy to compare across seeds
SOURCES = [
    {
        "path": "snapshots/20260703_004926_sta_lin_battery_only_price_sac",
        "eval-kind": "eval-ray",
        "label": "SAC",
        "color": "#75aef4",  # RL blue, matching snapshot_eval_violins.ipynb
    },
    {
        "path": "snapshots/20260704_144904_sta_lin_battery_only_price_ppo",
        "eval-kind": "eval-ray",
        "label": "PPO",
        "color": "#f1bd2d",  # RL orange, matching snapshot_eval_violins.ipynb
    },
    {
		"path": "snapshots/20260704_183007_sta_lin_battery_only_price_dreamerv3",
		"eval-kind": "eval-ray",
		"label": "DreamerV3",
		"color": "#f1652d",
	},
    {
        "path": "snapshots/20260705_234931_sta_lin_battery_only_price_rbc_base",
        "eval-kind": "eval-rbc",
        "rbc-strategy": "price_median_scaled_0.25",
        "label": "RBC \n(price_median_scaled_0.25)",
        "color": "#89f0cc",  # RBC green, matching snapshot_eval_violins.ipynb
    },
]
# Rule-based strategies available in the RBC snapshot above (pick one for "rbc-strategy"):
# ['deficit_discharge', 'do_nothing', 'price_median', 'price_median_autarky',
#  'price_median_scaled_0.25', 'price_median_scaled_0.5', 'price_median_scaled_0.75',
#  'pv_surplus_charge', 'self_coverage']

# Seeds to exclude from every figure and table below (applied after discovery, to both
# sources). Leave empty to keep all discovered seeds, e.g. EXCLUDE_SEEDS = [140, 141].
EXCLUDE_SEEDS: list[int] = [43, 44, 46, 70, 140]

METRICS = ("total_reward", "cum_E_kWh", "cum_price_EUR")


In [102]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

EVAL_KINDS = ("eval-ray", "eval-sb", "eval-rbc")


# Walk upwards from the cwd until the directory containing snapshots/ is found, so the
# notebook runs both from plotting/ and from the repo root.
def _find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "snapshots").is_dir():
            return candidate
    raise FileNotFoundError("No 'snapshots/' directory found upwards of the cwd — run inside the repo.")


REPO_ROOT = _find_repo_root()


# Maps a runs/eval_* directory to its eval kind — mirrors the run-id scheme
# tools/snapshot/submit_snapshot.py builds from --kind.
def _run_dir_eval_kind(run_dir: Path) -> str:
    if run_dir.name.startswith("eval_rbc_"):
        return "eval-rbc"
    if run_dir.name.startswith("eval_sb_"):
        return "eval-sb"
    return "eval-ray"


# Ray/SB3 eval CSV's sidecar JSON (same stem) carries the actual trial.seed this run used.
def _sidecar_seed(csv_path: Path) -> int | None:
    sidecar = csv_path.with_suffix(".json")
    if not sidecar.is_file():
        return None
    try:
        return json.loads(sidecar.read_text()).get("seed")
    except (json.JSONDecodeError, OSError):
        return None


# Rule-based episodes.csv's sibling args.json (one level above the per-strategy dir) records resolved_seed.
def _rbc_resolved_seed(csv_path: Path) -> int | None:
    args_path = csv_path.parent.parent / "args.json"
    if not args_path.is_file():
        return None
    try:
        return json.loads(args_path.read_text()).get("resolved_seed")
    except (json.JSONDecodeError, OSError):
        return None


# Discovers one record per seed-run for a single source: its CSV path and the actual seed
# read from that run's own metadata (never parsed from the directory name).
def _discover_source_records(source: dict) -> list[dict]:
    eval_kind = source["eval-kind"]
    if eval_kind not in EVAL_KINDS:
        raise ValueError(f"'eval-kind' must be one of {EVAL_KINDS}, got {eval_kind!r}")

    snapshot_dir = Path(source["path"])
    snapshot_dir = snapshot_dir if snapshot_dir.is_absolute() else REPO_ROOT / snapshot_dir
    if not snapshot_dir.is_dir():
        raise FileNotFoundError(f"Snapshot directory not found: {snapshot_dir}")

    run_dirs = [
        run_dir for run_dir in sorted((snapshot_dir / "runs").glob("eval_*"))
        if _run_dir_eval_kind(run_dir) == eval_kind
    ]
    if not run_dirs:
        raise FileNotFoundError(f"No '{eval_kind}' run directories found under {snapshot_dir / 'runs'}")

    records = []
    for run_dir in run_dirs:
        if eval_kind == "eval-rbc":
            strategy = source.get("rbc-strategy")
            for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/episodes.csv")):
                if strategy is not None and csv_path.parent.name != strategy:
                    continue
                records.append({
                    "source": source["label"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "seed": _rbc_resolved_seed(csv_path), "csv": csv_path,
                })
        else:
            for csv_path in sorted(run_dir.glob("eval_results/*_eval/eval_*.csv")):
                records.append({
                    "source": source["label"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "seed": _sidecar_seed(csv_path), "csv": csv_path,
                })
    if not records:
        raise FileNotFoundError(
            f"No eval CSVs discovered for source '{source['label']}' ({eval_kind}) under {snapshot_dir}"
        )
    return records


source_labels = [source["label"] for source in SOURCES]
if len(source_labels) != len(set(source_labels)):
    raise ValueError(f"Every source needs a unique 'label' (it names the per-seed violin group); got {source_labels}")
source_palette = {source["label"]: source["color"] for source in SOURCES}

eval_records: list[dict] = []
for source in SOURCES:
    eval_records.extend(_discover_source_records(source))

print(f"Repo root: {REPO_ROOT}")
for source in SOURCES:
    records = [record for record in eval_records if record["source"] == source["label"]]
    strategy = f", strategy={source.get('rbc-strategy')}" if source["eval-kind"] == "eval-rbc" else ""
    print(f"\nSource '{source['label']}' ({source['eval-kind']}{strategy}): {source['path']}")
    for record in sorted(records, key=lambda record: (record["seed"] is None, record["seed"])):
        print(f"  seed {str(record['seed']):>5}  <-  {record['run']}")


Repo root: /hkfs/home/haicore/iai/dj0397/AdvBuildingGym

Source 'SAC' (eval-ray): snapshots/20260703_004926_sta_lin_battery_only_price_sac
  seed    42  <-  eval_20260705_231614
  seed    43  <-  eval_20260706_104547_seed43
  seed    44  <-  eval_20260706_105216_seed44
  seed    45  <-  eval_20260706_105449_seed45
  seed    46  <-  eval_20260706_105502_seed46
  seed    70  <-  eval_20260706_105510_seed70
  seed    71  <-  eval_20260706_112012_seed71
  seed    91  <-  eval_20260729_170821_seed91
  seed   103  <-  eval_20260729_170822_seed103
  seed   140  <-  eval_20260706_111954_seed140
  seed   141  <-  eval_20260706_112005_seed141

Source 'PPO' (eval-ray): snapshots/20260704_144904_sta_lin_battery_only_price_ppo
  seed    42  <-  eval_20260705_231616
  seed    43  <-  eval_20260706_113732_seed43
  seed    44  <-  eval_20260706_113733_seed44
  seed    45  <-  eval_20260706_113734_seed45
  seed    46  <-  eval_20260706_113735_seed46
  seed    70  <-  eval_20260706_113737_seed70
  seed 

## Output naming

Everything saved here lands next to the notebook as `<date>_<time>_out_<i>.<file format>`:
one timestamp per notebook run, `out_index` counting up per saved artefact, so no run
overwrites an earlier one.

In [103]:
from datetime import datetime
from pathlib import Path

# VS Code exposes the notebook path; a plain Jupyter kernel starts in the notebook's directory.
OUT_DIR = Path(globals()["__vsc_ipynb_file__"]).parent if "__vsc_ipynb_file__" in globals() else Path.cwd()
OUT_RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
out_index = 0  # incremented once per saved artefact; reset by re-running this cell


def next_out_path(file_format: str) -> Path:
    """Next '<date>_<time>_out_<i>.<file_format>' path in this notebook's directory."""
    global out_index
    out_index += 1
    return OUT_DIR / f"{OUT_RUN_STAMP}_out_{out_index}.{file_format}"


## Align the seeds

One record per (source, seed). Runs whose seed cannot be resolved from metadata are dropped
first, then `EXCLUDE_SEEDS` is applied. The x-axis is the sorted union of the surviving seeds;
warnings are printed when a source has no run for one of them (seaborn leaves a gap there) and
when a source has several runs for the same seed (their episodes pool into one violin).

In [104]:
_missing_seed = [record for record in eval_records if record["seed"] is None]
for record in _missing_seed:
    print(f"WARNING: could not resolve the seed for {record['csv']} — dropped")
eval_records = [record for record in eval_records if record["seed"] is not None]
if not eval_records:
    raise FileNotFoundError("No eval CSVs with a resolvable seed were discovered.")

# Drop any explicitly excluded seeds (EXCLUDE_SEEDS) from every figure and table below.
_excluded_set = set(EXCLUDE_SEEDS)
_dropped = sorted({record["seed"] for record in eval_records if record["seed"] in _excluded_set})
if _dropped:
    print(f"Excluding seed(s) {_dropped} (EXCLUDE_SEEDS).")
eval_records = [record for record in eval_records if record["seed"] not in _excluded_set]
if not eval_records:
    raise ValueError(f"EXCLUDE_SEEDS={EXCLUDE_SEEDS} removed every discovered run — nothing left to plot.")

# One violin per (source, seed): flag duplicate seeds within a source (their episodes
# would pool into a single violin for that seed).
for source in SOURCES:
    seeds = [record["seed"] for record in eval_records if record["source"] == source["label"]]
    duplicates = sorted({seed for seed in seeds if seeds.count(seed) > 1})
    if duplicates:
        print(f"WARNING: source '{source['label']}' has multiple runs for seed(s) {duplicates} — "
              "their episodes are pooled into one violin per seed.")

# x-axis: the sorted union of seeds; warn where a source is not seed-aligned with the union.
seed_order = sorted({record["seed"] for record in eval_records})
for source in SOURCES:
    source_seeds = {record["seed"] for record in eval_records if record["source"] == source["label"]}
    missing = [seed for seed in seed_order if seed not in source_seeds]
    if missing:
        print(f"WARNING: source '{source['label']}' has no run for seed(s) {missing} — "
              "those seeds show only the other source's violin.")


def _display_path(csv_path: Path) -> str:
    try:
        return str(csv_path.relative_to(REPO_ROOT))
    except ValueError:
        return str(csv_path)


pd.DataFrame(
    {"source": record["source"], "seed": record["seed"], "run": record["run"], "csv": _display_path(record["csv"])}
    for record in sorted(eval_records, key=lambda record: (record["seed"], source_labels.index(record["source"])))
)


Excluding seed(s) [43, 44, 46, 70, 140] (EXCLUDE_SEEDS).


,source,seed,run,csv
0,SAC,42,eval_20260705_231614,snapshots/20260703_004926_sta_lin_battery_only...
1,PPO,42,eval_20260705_231616,snapshots/20260704_144904_sta_lin_battery_only...
2,DreamerV3,42,eval_20260706_115302_seed42,snapshots/20260704_183007_sta_lin_battery_only...
3,RBC \n(price_median_scaled_0.25),42,eval_rbc_20260705_234932,snapshots/20260705_234931_sta_lin_battery_only...
4,SAC,45,eval_20260706_105449_seed45,snapshots/20260703_004926_sta_lin_battery_only...
5,PPO,45,eval_20260706_113734_seed45,snapshots/20260704_144904_sta_lin_battery_only...
6,DreamerV3,45,eval_20260706_115305_seed45,snapshots/20260704_183007_sta_lin_battery_only...
7,RBC \n(price_median_scaled_0.25),45,eval_rbc_20260706_133840_seed45,snapshots/20260705_234931_sta_lin_battery_only...
8,SAC,71,eval_20260706_112012_seed71,snapshots/20260703_004926_sta_lin_battery_only...
9,PPO,71,eval_20260706_113738_seed71,snapshots/20260704_144904_sta_lin_battery_only...


## Load per-episode metrics

The CSVs are concatenated into one frame with a row per episode and the three metrics as
columns, each row tagged with its `source` and `seed`. A CSV missing a metric column is simply
absent from that metric's figure.

In [105]:
frames = []
for record in eval_records:
    csv_frame = pd.read_csv(record["csv"])
    present = [metric for metric in METRICS if metric in csv_frame.columns]
    missing = [metric for metric in METRICS if metric not in csv_frame.columns]
    if missing:
        print(f"note: source '{record['source']}' seed {record['seed']} lacks {missing} — skipped in those figures")
    frame = csv_frame[present].copy()
    frame["source"] = record["source"]
    frame["seed"] = record["seed"]
    frames.append(frame)

episodes = pd.concat(frames, ignore_index=True)
present_metrics = [metric for metric in METRICS if metric in episodes.columns]

episodes.groupby(["seed", "source"], sort=True).agg(
    episodes=("seed", "size"),
    **{f"mean_{metric}": (metric, "mean") for metric in present_metrics},
)


episodes  mean_total_reward  \
seed source                                                          
42   DreamerV3                               10          13.552012   
     PPO                                     10          13.471870   
     RBC \n(price_median_scaled_0.25)        10           9.735845   
     SAC                                     10          14.235960   
45   DreamerV3                               10          13.397464   
     PPO                                     10          12.312675   
     RBC \n(price_median_scaled_0.25)        10           9.374885   
     SAC                                     10          13.068847   
71   DreamerV3                               10          14.615034   
     PPO                                     10          14.371391   
     RBC \n(price_median_scaled_0.25)        10          10.852509   
     SAC                                     10          15.020272   
91   DreamerV3                               10          11.965888   
     PPO                                     10          11.635964   
     RBC \n(price_median_scaled_0.25)        10           8.389135   
     SAC                                     10          12.382234   
103  DreamerV3                               10          13.662983   
     PPO                                     10          13.492842   
     RBC \n(price_median_scaled_0.25)        10           9.826010   
     SAC                                     10          14.281120   
141  DreamerV3                               10          12.029902   
     PPO                                     10          12.395909   
     RBC \n(price_median_scaled_0.25)        10           7.744611   
     SAC                                     10          12.884280   

                                       mean_cum_E_kWh  mean_cum_price_EUR  
seed source                                                                
42   DreamerV3                              46.751349           -3.765060  
     PPO                                    46.765021           -3.618641  
     RBC \n(price_median_scaled_0.25)       42.604213           -2.698555  
     SAC                                    42.087546           -3.463104  
45   DreamerV3                              46.355331           -4.414143  
     PPO                                    46.369728           -3.820409  
     RBC \n(price_median_scaled_0.25)       42.690142           -3.193831  
     SAC                                    41.669323           -3.802764  
71   DreamerV3                              48.097437           -4.503316  
     PPO                                    48.098967           -4.381068  
     RBC \n(price_median_scaled_0.25)       44.034734           -3.397704  
     SAC                                    43.578278           -4.092103  
91   DreamerV3                              49.484650           -4.124740  
     PPO                                    49.515037           -3.859273  
     RBC \n(price_median_scaled_0.25)       45.659651           -3.005038  
     SAC                                    45.297014           -3.699635  
103  DreamerV3                              45.080695           -3.612205  
     PPO                                    45.080695           -3.726638  
     RBC \n(price_median_scaled_0.25)       41.129047           -2.706206  
     SAC                                    39.936927           -3.314926  
141  DreamerV3                              46.027795           -3.553807  
     PPO                                    46.061401           -3.713324  
     RBC \n(price_median_scaled_0.25)       41.522957           -2.418653  
     SAC                                    42.861213           -3.415747

## Violin plots

At each seed the sources are drawn as **dodged violins side by side**, each in its `SOURCES`
colour, with the individual episodes overlaid as dodged dots. Every violin carries a red
**mean** dot and a dashed **median** line. Each figure is saved next to this notebook as
`<date>_<time>_out_<i>.pdf`.

The legend sits beside the axes for up to `LEGEND_BELOW_ABOVE_SEEDS` (4) seeds; with more seeds
it moves **under the plot**, spread over enough columns to stay within `LEGEND_MAX_ROWS` (2)
rows.

The figures carry no title — the thesis caption names them.

In [ ]:
from math import ceil

import numpy as np
from matplotlib.lines import Line2D

sns.set_theme(style="whitegrid", context="notebook")

METRIC_LABELS = {
    "total_reward": "Total reward per episode",
    "cum_E_kWh": "Net grid exchange [kWh]",
    "cum_price_EUR": "Cumulative energy price [EUR]",
}

# Stat markers drawn on top of each violin (replaces seaborn's faint inner box)
MEAN_COLOR = "#e34948"  # red dot
MEDIAN_COLOR = "#1a1a19"  # narrow dashed line
MEDIAN_DASH = (0, (4, 2))

# Legend placement: beside the axes while the x-axis is narrow, under the plot as soon as more
# than LEGEND_BELOW_ABOVE_SEEDS seeds are drawn (a wide figure wastes the side strip).
LEGEND_BELOW_ABOVE_SEEDS = 4
LEGEND_MAX_ROWS = 2  # a legend under the plot never grows past this many rows
LEGEND_BELOW_PAD = 0.03  # axes-fraction gap between the x-axis label and the legend below it
LEGEND_FONT_SIZE = 16  # point size of the legend entry labels

# Horizontal packing, in units of one seed slot (adjacent seeds sit 1.0 apart):
#   DODGE_WIDTH      the slot the sources share, passed to violinplot as `width`. The remainder
#                    (1 - DODGE_WIDTH) is the gap between two seed groups, so a value close to 1
#                    packs the groups tightly.
#   STRIP_DODGE_WIDTH  seaborn hardcodes the strip dodge spread at 0.8 * the slot width
#                    (plot_strips in seaborn/categorical.py) and offers no parameter for it, so
#                    the episode dots are rescaled by DODGE_WIDTH / STRIP_DODGE_WIDTH afterwards.
#   X_AXIS_MARGIN    space left between the axes edges and the outermost violins.
DODGE_WIDTH = 0.85
STRIP_DODGE_WIDTH = 0.8
X_AXIS_MARGIN = 0.02

_seed_position = {seed: index for index, seed in enumerate(seed_order)}


# Centre offset of a source's dodged violin within a seed's slot, so the stat markers land
# on the exact violin seaborn draws for that (seed, source). Matches seaborn's even hue split.
def _dodge_offset(source_label: str) -> float:
    hue_index = source_labels.index(source_label)
    return DODGE_WIDTH * ((hue_index + 0.5) / len(source_labels) - 0.5)


# Re-centres the stripplot dots on their violins: seaborn spread them over STRIP_DODGE_WIDTH, so
# every dot's offset from its seed's centre (dodge + jitter) is scaled to DODGE_WIDTH.
def _match_strip_dodge(collections: list) -> None:
    scale = DODGE_WIDTH / STRIP_DODGE_WIDTH
    for collection in collections:
        offsets = np.asarray(collection.get_offsets(), dtype=float)
        if offsets.size == 0:
            continue
        seed_centres = np.round(offsets[:, 0])  # dodge + jitter stay well inside +/- 0.5
        offsets[:, 0] = seed_centres + (offsets[:, 0] - seed_centres) * scale
        collection.set_offsets(offsets)


# Overlays each dodged violin's mean (red dot) and median (narrow dashed line).
def _draw_stat_markers(ax, data: pd.DataFrame, metric: str) -> None:
    half_width = 0.9 * (DODGE_WIDTH / len(source_labels)) / 2
    stats = data.groupby(["seed", "source"])[metric].agg(["mean", "median"])
    for (seed, source), row in stats.iterrows():
        if seed not in _seed_position or source not in source_labels:
            continue
        x_position = _seed_position[seed] + _dodge_offset(source)
        ax.hlines(
            row["median"], x_position - half_width, x_position + half_width,
            color=MEDIAN_COLOR, linewidth=0.9, linestyle=MEDIAN_DASH, zorder=5,
        )
        ax.scatter(
            x_position, row["mean"],
            color=MEAN_COLOR, s=18, edgecolor="white", linewidth=0.5, zorder=6,
        )


# Axes-fraction y for the top edge of a legend placed under the plot: just below the x-axis's
# rendered footprint (tick labels + "Seed" label), so the legend cannot overdraw them. Negative,
# because it sits below the axes' bottom edge. Only valid once the axes box is final.
def _legend_top_below_axes(ax) -> float:
    renderer = ax.figure.canvas.get_renderer()
    axis_bbox = ax.xaxis.get_tightbbox(renderer)  # display px; None when the axis draws nothing
    axes_bbox = ax.get_window_extent(renderer)
    if axis_bbox is None:
        return -LEGEND_BELOW_PAD
    return (axis_bbox.y0 - axes_bbox.y0) / axes_bbox.height - LEGEND_BELOW_PAD


# Places the legend outside the axes: to the right for a narrow x-axis, but under the plot once
# the seed fan-out makes the figure wide. Below the plot the entries go on one row when that row
# fits the figure width, else on `ceil(n / LEGEND_MAX_ROWS)` columns — so the legend never
# exceeds LEGEND_MAX_ROWS rows.
def _draw_legend(ax, handles: list, labels: list[str]) -> None:
    if len(seed_order) <= LEGEND_BELOW_ABOVE_SEEDS:
        ax.legend(handles=handles, labels=labels, title="", frameon=False,
                  loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0,
                  fontsize=LEGEND_FONT_SIZE)
        return

    anchor_y = _legend_top_below_axes(ax)
    figure_width = ax.figure.get_window_extent().width
    column_counts = dict.fromkeys([len(handles), ceil(len(handles) / LEGEND_MAX_ROWS)])  # widest first
    for index, columns in enumerate(column_counts):
        legend = ax.legend(handles=handles, labels=labels, title="", frameon=False, loc="upper center",
                           bbox_to_anchor=(0.5, anchor_y), borderaxespad=0.0, ncol=columns,
                           fontsize=LEGEND_FONT_SIZE)
        legend_width = legend.get_window_extent(ax.figure.canvas.get_renderer()).width
        if legend_width <= figure_width or index == len(column_counts) - 1:
            return  # fits the figure width, or this is the LEGEND_MAX_ROWS-row fallback


# Draws one violin per source at each seed for `metric` and exports the figure as a PDF
# next to this notebook. The figures carry no title — the thesis caption names them.
def violin_for_metric(metric: str) -> None:
    data = episodes.dropna(subset=[metric]) if metric in episodes.columns else episodes.iloc[0:0]
    if data.empty:
        print(f"no data for '{metric}' — figure skipped")
        return

    figure_width = max(8.0, 0.9 * len(seed_order) * len(source_labels))
    fig, ax = plt.subplots(figsize=(figure_width, 5.0))
    sns.violinplot(
        data=data,
        x="seed",
        y=metric,
        hue="source",
        order=seed_order,
        hue_order=source_labels,
        palette=source_palette,
        dodge=True,  # two violins side by side per seed
        width=DODGE_WIDTH,
        density_norm="area",
        inner=None,  # the stat markers below replace seaborn's faint inner box
        linewidth=1.0,
        saturation=1.0,
        ax=ax,
    )
    violin_collections = list(ax.collections)
    sns.stripplot(
        data=data, x="seed", y=metric, hue="source",
        order=seed_order, hue_order=source_labels, dodge=True,
        palette={source: "0.1" for source in source_labels},
        size=3, alpha=0.6, jitter=0.08, legend=False, ax=ax,
    )
    _match_strip_dodge([c for c in ax.collections if c not in violin_collections])
    _draw_stat_markers(ax, data, metric)

    # Trim the padding seaborn leaves left of the first and right of the last violin.
    ax.set_xlim(-DODGE_WIDTH / 2 - X_AXIS_MARGIN, len(seed_order) - 1 + DODGE_WIDTH / 2 + X_AXIS_MARGIN)
    ax.set_xlabel("Seed")
    ax.set_ylabel(METRIC_LABELS.get(metric, metric))
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)
    fig.tight_layout()  # settles the axes box first — _draw_legend measures against it

    # Legend: the source colours seaborn built + the mean/median stat markers, outside the axes.
    source_handles, source_legend_labels = ax.get_legend_handles_labels()
    stat_handles = [
        Line2D([], [], color=MEAN_COLOR, marker="o", linestyle="none", markersize=5,
               markeredgewidth=0.5, label="mean"),
        Line2D([], [], color=MEDIAN_COLOR, linewidth=0.9, linestyle=MEDIAN_DASH, label="median"),
    ]
    _draw_legend(
        ax,
        source_handles + stat_handles,
        source_legend_labels + [handle.get_label() for handle in stat_handles],
    )

    pdf_path = next_out_path("pdf")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"saved {pdf_path}  ({metric})")
    plt.show()


In [ ]:
violin_for_metric("total_reward")


In [ ]:
violin_for_metric("cum_E_kWh")


In [ ]:
violin_for_metric("cum_price_EUR")


## Per-seed statistics, per source

For each `(source, seed)` the per-episode metrics are summarised (`mean` / `std` / `median` /
`min` / `max`) over that seed's episodes — one row per seed, **not** collapsed across seeds.
Reading a source's rows top to bottom shows how its held-out performance shifts from seed to
seed, while the per-row `std` is the episode spread inside a single seed. Comparing one seed
across sources shows which source wins on that exact draw. The table is also written to
`<date>_<time>_out_<i>.csv`.

In [110]:
present_metrics = [metric for metric in METRICS if metric in episodes.columns]

# Per-(source, seed) statistics of the per-episode metrics: one row per seed instead of
# collapsing the seeds together, so each source's held-out performance can be read seed by
# seed. Here `std`/`min`/`max` describe the spread of that seed's episode draws (10 per run,
# 20 where a seed pooled two runs — see the DreamerV3 seed-42 warning above), so a wide std
# means that single seed's episodes are themselves spread out.
per_seed_stats = (
    episodes.groupby(["source", "seed"])[present_metrics]
    .agg(["mean", "std", "median", "min", "max"])
    .sort_index()
)
per_seed_stats.insert(0, ("episodes", ""), episodes.groupby(["source", "seed"]).size())
per_seed_stats.to_csv(next_out_path("csv"), float_format="%.2f", index=True)
per_seed_stats


episodes total_reward            \
                                                       mean       std   
source                           seed                                   
DreamerV3                        42         10    13.552012  2.823790   
                                 45         10    13.397464  2.727784   
                                 71         10    14.615034  3.139797   
                                 91         10    11.965888  3.119124   
                                 103        10    13.662983  2.969126   
                                 141        10    12.029902  5.582282   
PPO                              42         10    13.471870  4.032283   
                                 45         10    12.312675  4.147067   
                                 71         10    14.371391  4.007829   
                                 91         10    11.635964  3.796265   
                                 103        10    13.492842  3.580648   
                                 141        10    12.395909  5.308375   
RBC \n(price_median_scaled_0.25) 42         10     9.735845  3.644916   
                                 45         10     9.374885  2.416797   
                                 71         10    10.852509  3.743984   
                                 91         10     8.389135  3.452870   
                                 103        10     9.826010  3.504947   
                                 141        10     7.744611  4.659296   
SAC                              42         10    14.235960  3.074803   
                                 45         10    13.068847  2.642311   
                                 71         10    15.020272  4.260532   
                                 91         10    12.382234  3.480828   
                                 103        10    14.281120  3.951770   
                                 141        10    12.884280  6.320858   

                                                                        \
                                          median        min        max   
source                           seed                                    
DreamerV3                        42    13.352186   8.858986  17.384323   
                                 45    12.334752   9.272996  17.336435   
                                 71    14.916447   9.174531  21.112291   
                                 91    12.095743   5.699593  16.908395   
                                 103   14.054954   7.922276  17.549864   
                                 141   12.427159   3.461318  20.575107   
PPO                              42    14.024971   5.317211  18.388013   
                                 45    12.843423   5.317211  17.834307   
                                 71    15.243778   7.533430  21.210184   
                                 91    11.377833   5.866306  18.477021   
                                 103   12.537304   6.987673  18.243098   
                                 141   12.517495   4.844083  21.301516   
RBC \n(price_median_scaled_0.25) 42     8.881057   5.172132  16.717488   
                                 45     9.434769   5.631891  13.338427   
                                 71    11.061664   2.839784  18.102803   
                                 91     8.777751   1.569077  15.125715   
                                 103    9.800594   3.491666  15.498744   
                                 141    7.314608   1.747698  14.939385   
SAC                              42    13.404769  10.192123  20.053379   
                                 45    12.060947   9.754510  17.116785   
                                 71    15.131567   9.307114  24.852340   
                                 91    11.760358   8.173938  18.606752   
                                 103   13.045183   8.608085  20.687518   
                                 141   12.685589   3.509996  23.152897   

                                       cum_E_kWh                   

## Variance across seeds, per source

Collapses the per-seed panel above into one row per source. `mean` / `std` / `median` / `min` /
`max` summarise each source's **per-seed means**, so `std` is the seed-to-seed spread of the
mean (small = consistent across the differently seeded held-out draws). Two extra columns
summarise the *other* per-seed statistics:

- **`std_of_medians`** — seed-to-seed spread of the per-seed **median**, a robust twin of `std`.
- **`mean_of_stds`** — the per-seed within-episode **std** averaged over seeds, i.e. the typical
  episode-to-episode spread inside one seed.

`std` and `std_of_medians` are `NaN` when only a single seed survives the filters. The table is
written to `<date>_<time>_out_<i>.csv`.

**Result:** mean total reward across seeds is 13.65 (SAC), 13.20 (DreamerV3) and 12.95 (PPO)
against 9.32 for the rule-based baseline, with a comparable seed-to-seed `std` of about 1.0-1.1
for all four — the RL advantage holds at every individual seed.

In [111]:
present_metrics = [metric for metric in METRICS if metric in episodes.columns]

# Per-seed central tendency + spread of each metric: one row per (source, seed).
per_seed = episodes.groupby(["source", "seed"])[present_metrics].agg(["mean", "median", "std"])
seed_means = per_seed.xs("mean", axis=1, level=1)      # per-seed mean,   index (source, seed)
seed_medians = per_seed.xs("median", axis=1, level=1)  # per-seed median, index (source, seed)
seed_stds = per_seed.xs("std", axis=1, level=1)        # per-seed std,    index (source, seed)

# Collapse across seeds, per source. The mean/std/median/min/max columns summarise the
# per-seed MEANS (so `std` is the seed-to-seed spread of the mean). On top of that:
#   std_of_medians  seed-to-seed spread of the per-seed MEDIAN (robust twin of `std`)
#   mean_of_stds    the per-seed within-episode STD, averaged over seeds (typical episode spread)
# `std`/`std_of_medians` are NaN when only a single seed survives the filters.
summaries = {
    "mean": seed_means.groupby("source").mean(),
    "std": seed_means.groupby("source").std(),
    "median": seed_means.groupby("source").median(),
    "min": seed_means.groupby("source").min(),
    "max": seed_means.groupby("source").max(),
    "std_of_medians": seed_medians.groupby("source").std(),
    "mean_of_stds": seed_stds.groupby("source").mean(),
}
across_seed_stats = pd.concat(summaries, axis=1)
across_seed_stats.columns = across_seed_stats.columns.swaplevel(0, 1)  # -> (metric, summary)
across_seed_stats = across_seed_stats.reindex(columns=present_metrics, level=0)
across_seed_stats = across_seed_stats.reindex(columns=list(summaries), level=1)
across_seed_stats.to_csv(next_out_path("csv"), float_format="%.2f", index=True)
across_seed_stats


total_reward                                  \
                                         mean       std     median        min   
source                                                                          
DreamerV3                           13.203880  1.026806  13.474738  11.965888   
PPO                                 12.946775  1.002717  12.933889  11.635964   
RBC \n(price_median_scaled_0.25)     9.320499  1.107152   9.555365   7.744611   
SAC                                 13.645452  1.014936  13.652403  12.382234   

                                                                         \
                                        max std_of_medians mean_of_stds   
source                                                                    
DreamerV3                         14.615034       1.119425     3.393650   
PPO                               14.371391       1.351923     4.145411   
RBC \n(price_median_scaled_0.25)  10.852509       1.242217     3.570468   
SAC                               15.020272       1.202127     3.955184   

                                  cum_E_kWh                       ...  \
                                       mean       std     median  ...   
source                                                            ...   
DreamerV3                         46.966210  1.579705  46.553340  ...   
PPO                               46.981808  1.584191  46.567375  ...   
RBC \n(price_median_scaled_0.25)  42.940124  1.676764  42.647178  ...   
SAC                               42.571717  1.817420  42.474379  ...   

                                                                         \
                                        max std_of_medians mean_of_stds   
source                                                                    
DreamerV3                         49.484650       3.144582     9.343366   
PPO                               49.515037       3.119211     9.343274   
RBC \n(price_median_scaled_0.25)  45.659651       2.953490     9.633847   
SAC                               45.297014       1.962519     8.111477   

                                 cum_price_EUR                                \
                                          mean       std    median       min   
source                                                                         
DreamerV3                            -3.995545  0.411084 -3.944900 -4.503316   
PPO                                  -3.853225  0.272161 -3.773523 -4.381068   
RBC \n(price_median_scaled_0.25)     -2.903331  0.361896 -2.855622 -3.397704   
SAC                                  -3.631380  0.290260 -3.581369 -4.092103   

                                                                        
                                       max std_of_medians mean_of_stds  
source                                                                  
DreamerV3                        -3.553807       0.504694     1.987989  
PPO                              -3.618641       0.485668     1.576758  
RBC \n(price_median_scaled_0.25) -2.418653       0.480207     1.733237  
SAC                              -3.314926       0.442194     1.862517  

[4 rows x 21 columns]

## Pooled return statistics across all seeds, per source

Same one-row-per-source shape as the table above, but for the **return** (`total_reward`) only
and computed **without any per-seed aggregation**: every episode of every surviving seed enters
as one sample, so `mean` / `std` / `median` describe the raw pooled return distribution.

This is deliberately *not* the same quantity as the table above:

- **above** — each seed is first collapsed to its mean, then those per-seed means are
  summarised, so `std` is the seed-to-seed spread of the mean (each seed weighs the same, the
  within-seed episode spread is invisible).
- **here** — no intermediate step: `mean` is the grand mean over all episodes and `std` the
  total spread of individual episode returns, mixing within-seed variation with the
  between-seed shift, so it is markedly larger than the `std` above.

Because the pooling is over episodes, a seed contributing more episodes weighs more; the
`seeds` / `episodes` / `min` / `max` columns that would make this visible are commented out in
the cell below and can be re-enabled. The table is written to `<date>_<time>_out_<i>.csv`.

In [112]:
RETURN_METRIC = "total_reward"

if RETURN_METRIC not in episodes.columns:
    raise KeyError(f"'{RETURN_METRIC}' is absent from every loaded eval CSV — nothing to pool.")

# Pool the per-episode returns of all seeds of a source into ONE sample set, then summarise
# it directly — no per-seed mean/median is formed on the way, so `std` is the spread of the
# individual episode returns (within-seed + between-seed), not the spread of seed averages.
pooled_returns = episodes.dropna(subset=[RETURN_METRIC])

pooled_return_stats = pooled_returns.groupby("source").agg(
    # seeds=("seed", "nunique"),
    # episodes=(RETURN_METRIC, "size"),
    mean=(RETURN_METRIC, "mean"),
    std=(RETURN_METRIC, "std"),  # sample std (ddof=1) over all pooled episodes
    median=(RETURN_METRIC, "median"),
    # min=(RETURN_METRIC, "min"),
    # max=(RETURN_METRIC, "max"),
)
# Row order follows SOURCES rather than the alphabetical groupby order, so the table reads
# in the same order as the violin legend.
pooled_return_stats = pooled_return_stats.reindex(
    [label for label in source_labels if label in pooled_return_stats.index]
)
pooled_return_stats.to_csv(next_out_path("csv"), float_format="%.2f", index=True)
pooled_return_stats

,mean,std,median
source,,,
SAC,13.645452,4.058832,12.662219
PPO,12.946775,4.105922,13.165997
DreamerV3,13.203880,3.511558,13.384106
RBC \n(price_median_scaled_0.25),9.320499,3.619221,9.257426
